In [1]:

import numpy as np

from qiskit.transpiler.passes.routing.commuting_2q_gate_routing import SwapStrategy

from qiskit_aer import AerSimulator
from qiskit_aer.primitives import SamplerV2 as Sampler


from qiskit_qaoa.utils.circuit_graph_utils import circuit_construction
from qiskit_qaoa.utils.hamiltonian_utils import get_Q_and_hamiltonian
from qiskit_qaoa.utils.string_utils import evaluate_sparse_pauli_samples
from qiskit_qaoa.utils.logging import get_logger

logger = get_logger(__name__)

backend_options = dict(
    # method='matrix_product_state',
    # matrix_product_state_max_bond_dimension='20', 
    method='statevector',
    device='CPU',
    precision='single',
    basis_gates = ['rx', 'ry', 'rz', 'cx']
)
# fake_fez = FakeFez()
backend = AerSimulator(**backend_options)
sampler = Sampler.from_backend(backend)

filename = 'test_N2_W2'

rng = np.random.default_rng()

data_file = f'/lustre/scratch127/qpg/jc59/new_qubo_formulation/oriented/qubo_data/qubo_data_{filename}.gfa.pkl'

Q, hamiltonian, offset, ising_offset = get_Q_and_hamiltonian(data_file)
num_qubits: int = hamiltonian.num_qubits

    
swap_strat = SwapStrategy.from_line(list(range(num_qubits)))
edge_coloring = {(idx, idx + 1): (idx + 1) % 2 for idx in range(num_qubits)}

singles = hamiltonian[hamiltonian.paulis.z.sum(axis=-1) == 1]
doubles = hamiltonian[hamiltonian.paulis.z.sum(axis=-1) == 2]

keys = [np.binary_repr(x, num_qubits) for x in range(2**num_qubits)]
evals = evaluate_sparse_pauli_samples(keys, hamiltonian) + ising_offset


def get_energy(qc):
    job = backend.run([qc],shots=1)
    sampler_result = job.result()
    data = sampler_result.results[0].data

    sv = np.asarray(data.statevector)
    energy = np.sum(np.abs(sv) ** 2 * evals)
    return energy


def LR_QAOA(p, deltab, deltag, circ):
    betas = [(1-k/p) * deltab for k in range(p)]
    gammas = [(k+1) / p * deltag for k in range(p)]
    fixed_params = betas + gammas
    
    if circ is None:
        circ_dict = circuit_construction(singles, doubles, None, swap_strat, edge_coloring, {}, p, None, None)
        circuit = circ_dict["circuit_to_sample"]
        logger.info(f'p = {p}. Circuit depth: {circuit.depth()}')
    else:
        circuit = circ

    fixed_param_bind = {circuit.parameters[i]: fixed_params[i] for i in range(2*p)}
    fixed_qc = circuit.assign_parameters(fixed_param_bind)
    fixed_qc.remove_final_measurements()
    fixed_qc.save_statevector()

    energy = get_energy(fixed_qc)
        
    print(deltab, deltag, p, energy)
    return energy, circuit
        




In [5]:

circuit = None
p = 10
db = 0.3
dg = 0.6
e, circuit = LR_QAOA(p, 0, 0, circuit)
e, circuit = LR_QAOA(p, -db, dg, circuit)
e, circuit = LR_QAOA(p, db, dg, circuit)

10:16:18 - __main__ - INFO - p = 10. Circuit depth: 322
0 0 10 46.49998891353673
-0.3 0.6 10 30.964610979975383
0.3 0.6 10 77.80940565663713


In [3]:
from qiskit import QuantumCircuit
qc = QuantumCircuit(8)
qc.x(0)
qc.x(6)
# qc.x(7-0)
# qc.x(7-6)
qc.save_statevector()
get_energy(qc)

np.float64(0.0)

In [12]:
X = np.array([[0,1],[1,0]])


In [13]:
from scipy.linalg import expm
expm(1j*np.pi/8*X)

array([[0.92387953+0.j        , 0.        +0.38268343j],
       [0.        +0.38268343j, 0.92387953+0.j        ]])

In [11]:
np.cos(np.pi/8) * np.eye(2) + np.sin(np.pi/8) * X

array([[0.92387953, 0.38268343],
       [0.38268343, 0.92387953]])

In [7]:
ubackend = AerSimulator(method='unitary')
qc = QuantumCircuit(1)
qc.rx(-2*np.pi/8, 0)
qc.save_unitary()
res = ubackend.run(qc).result()

In [10]:
res.data()['unitary']

Operator([[0.92387953+0.j        , 0.        +0.38268343j],
          [0.        +0.38268343j, 0.92387953+0.j        ]],
         input_dims=(2,), output_dims=(2,))
